In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import glob

glob.glob("/content/drive/MyDrive/Raw_Data_PM25/*")


['/content/drive/MyDrive/Raw_Data_PM25/PM25_2020.csv',
 '/content/drive/MyDrive/Raw_Data_PM25/PM25_2021.csv',
 '/content/drive/MyDrive/Raw_Data_PM25/PM25_2022.csv',
 '/content/drive/MyDrive/Raw_Data_PM25/PM25_2023.csv',
 '/content/drive/MyDrive/Raw_Data_PM25/en_climate_daily_AB_3031094_2020_P1D.csv',
 '/content/drive/MyDrive/Raw_Data_PM25/en_climate_daily_AB_3031094_2021_P1D.csv',
 '/content/drive/MyDrive/Raw_Data_PM25/en_climate_daily_AB_3031094_2022_P1D.csv',
 '/content/drive/MyDrive/Raw_Data_PM25/en_climate_daily_AB_3031094_2023_P1D.csv',
 '/content/drive/MyDrive/Raw_Data_PM25/Calgary_PM25_Weather_Cleaned_v3.csv']

In [ ]:
import pandas as pd
import numpy as np
import glob

# 3.1 – find all PM2.5 raw files (2020–2023)
pm_files = sorted(glob.glob("/content/drive/MyDrive/Raw_Data_PM25/PM25_20*.csv"))
print("PM files:", pm_files)

pm_list = []

for f in pm_files:
    # skip the 7 metadata rows at the top of each NAPS file
    df = pd.read_csv(f, skiprows=7, encoding="utf-8-sig")

    # detect the date column (e.g. "Date//Date")
    date_col = [c for c in df.columns if "Date" in c][0]
    df["date"] = pd.to_datetime(df[date_col])

    # detect all hourly columns (H01, H02, ..., H24)
    hour_cols = [c for c in df.columns if c.startswith("H")]

    # replace -999 with NaN and compute the mean over 24 hours
    df["PM25"] = df[hour_cols].replace(-999, np.nan).mean(axis=1)

    # keep only daily date + PM25
    pm_list.append(df[["date", "PM25"]])

# 3.2 – concatenate all years and sort
pm_raw = pd.concat(pm_list, ignore_index=True)
pm_raw = pm_raw.sort_values("date")

# 3.3 – aggregate to one value per day (in case multiple rows exist per day)
pm_daily = pm_raw.groupby("date", as_index=False)["PM25"].mean()

pm_daily.head()


PM files: ['/content/drive/MyDrive/Raw_Data_PM25/PM25_2020.csv', '/content/drive/MyDrive/Raw_Data_PM25/PM25_2021.csv', '/content/drive/MyDrive/Raw_Data_PM25/PM25_2022.csv', '/content/drive/MyDrive/Raw_Data_PM25/PM25_2023.csv']


,date,PM25
0,2020-01-01,5.080895
1,2020-01-02,4.965239
2,2020-01-03,6.367912
3,2020-01-04,4.359544
4,2020-01-05,3.706763


In [ ]:
# 4.1 – find all daily climate files
weather_files = sorted(glob.glob("/content/drive/MyDrive/Raw_Data_PM25/en_climate_daily_AB_3031094_*_P1D.csv"))
print("Weather files:", weather_files)

weather_list = []

for f in weather_files:
    df = pd.read_csv(f)

    # date column in these files is "Date/Time"
    df["date"] = pd.to_datetime(df["Date/Time"])

    # keep only date, mean temperature, and total precip
    df = df[["date", "Mean Temp (°C)", "Total Precip (mm)"]]

    weather_list.append(df)

weather_raw = pd.concat(weather_list, ignore_index=True)
weather_raw = weather_raw.sort_values("date")

weather_raw.head()


Weather files: ['/content/drive/MyDrive/Raw_Data_PM25/en_climate_daily_AB_3031094_2020_P1D.csv', '/content/drive/MyDrive/Raw_Data_PM25/en_climate_daily_AB_3031094_2021_P1D.csv', '/content/drive/MyDrive/Raw_Data_PM25/en_climate_daily_AB_3031094_2022_P1D.csv', '/content/drive/MyDrive/Raw_Data_PM25/en_climate_daily_AB_3031094_2023_P1D.csv']


,date,Mean Temp (°C),Total Precip (mm)
0,2020-01-01,-1.6,0.0
1,2020-01-02,-4.2,1.4
2,2020-01-03,-0.4,0.0
3,2020-01-04,2.4,0.6
4,2020-01-05,-4.1,0.1


In [ ]:
# 5 – Merge PM daily with Weather daily
df_merge = pd.merge(pm_daily, weather_raw, on="date", how="inner")

# 5.1 – Create lag and rolling features (same as original v2)
df_merge["PM25_lag1"] = df_merge["PM25"].shift(1)
df_merge["PM25_roll3"] = df_merge["PM25"].rolling(window=3).mean()

# 5.2 – Time-based features
df_merge["month"] = df_merge["date"].dt.month
df_merge["dow"] = df_merge["date"].dt.dayofweek

df_merge["season"] = df_merge["month"].apply(
    lambda m: "Winter" if m in [12, 1, 2]
    else "Spring" if m in [3, 4, 5]
    else "Summer" if m in [6, 7, 8]
    else "Fall"
)

df_merge.head()


,date,PM25,Mean Temp (°C),Total Precip (mm),PM25_lag1,PM25_roll3,month,dow,season
0,2020-01-01,5.080895,-1.6,0.0,NaN,NaN,1,2,Winter
1,2020-01-02,4.965239,-4.2,1.4,5.080895,NaN,1,3,Winter
2,2020-01-03,6.367912,-0.4,0.0,4.965239,5.471348,1,4,Winter
3,2020-01-04,4.359544,2.4,0.6,6.367912,5.230898,1,5,Winter
4,2020-01-05,3.706763,-4.1,0.1,4.359544,4.811406,1,6,Winter


In [ ]:
# --- 6.0 Rename columns BEFORE the second merge (must match v3) ---
df_merge = df_merge.rename(columns={
    "Mean Temp (°C)": "Mean Temp",
    "Total Precip (mm)": "Total Preci"
})

weather_raw_renamed = weather_raw.rename(columns={
    "Mean Temp (°C)": "Mean Temp",
    "Total Precip (mm)": "Total Preci"
})

# --- 6.1 Second merge to create _x and _y columns ---
final_df = pd.merge(
    df_merge,
    weather_raw_renamed,
    on="date",
    how="inner",
    suffixes=("_x", "_y")
)

# --- 6.2 Add remaining features ---
final_df["year"] = final_df["date"].dt.year
final_df["dayofyear"] = final_df["date"].dt.dayofyear

# --- 6.3 Reorder columns EXACTLY like v3 ---
final_df = final_df[
    [
        "date",
        "PM25",
        "Mean Temp_x",
        "Total Preci_x",
        "PM25_lag1",
        "PM25_roll3",
        "month",
        "dow",
        "season",
        "Mean Temp_y",
        "Total Preci_y",
        "year",
        "dayofyear"
    ]
]

final_df.head()


,date,PM25,Mean Temp_x,Total Preci_x,PM25_lag1,PM25_roll3,month,dow,season,Mean Temp_y,Total Preci_y,year,dayofyear
0,2020-01-01,5.080895,-1.6,0.0,NaN,NaN,1,2,Winter,-1.6,0.0,2020,1
1,2020-01-02,4.965239,-4.2,1.4,5.080895,NaN,1,3,Winter,-4.2,1.4,2020,2
2,2020-01-03,6.367912,-0.4,0.0,4.965239,5.471348,1,4,Winter,-0.4,0.0,2020,3
3,2020-01-04,4.359544,2.4,0.6,6.367912,5.230898,1,5,Winter,2.4,0.6,2020,4
4,2020-01-05,3.706763,-4.1,0.1,4.359544,4.811406,1,6,Winter,-4.1,0.1,2020,5


In [ ]:
# Fix column names to match the previous final dataset exactly
final_df = final_df.rename(columns={
    "Mean Temp_x": "Mean Temp (°C)_x",
    "Total Preci_x": "Total Precip (mm)_x",
    "Mean Temp_y": "Mean Temp (°C)_y",
    "Total Preci_y": "Total Precip (mm)_y"
})


In [ ]:
output_path = "/content/drive/MyDrive/Final_Cleaned_Data/Calgary_PM25_Weather_Cleaned_v3.csv"

final_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Final dataset saved at:", output_path)


Final dataset saved at: /content/drive/MyDrive/Final_Cleaned_Data/Calgary_PM25_Weather_Cleaned_v3.csv
